# Recycling Buddy — YOLO26n Fine-Tuning

ゴミ分類キオスク用にYOLO26nをファインチューニングするノートブック。

**ランタイム設定:** メニュー → ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択してください。

### データソース
- **TACO** (Trash Annotations in Context) — 1500枚, 60カテゴリ → 12クラスに統合
- **Open Images V5** — battery, food, coffee_cup, glass_bottle, tin_can, paper を補充

### 出力クラス (12)
```
0: plastic_bottle   → recycling
1: can              → recycling
2: paper_cup        → landfill
3: plastic_cup      → landfill
4: glass_bottle     → recycling
5: cardboard        → recycling
6: food_waste       → compost
7: paper            → recycling
8: plastic_bag      → landfill
9: plastic_container→ landfill
10: battery         → special
11: styrofoam       → landfill
```

## 1. 環境セットアップ

In [ ]:
!nvidia-smi
!pip install -q ultralytics

In [ ]:
import csv
import io
import json
import os
import random
import shutil
import urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

WORKDIR = Path("/content/training")
WORKDIR.mkdir(exist_ok=True)
os.chdir(WORKDIR)
print(f"Working directory: {WORKDIR}")

## 2. TACO データセットのダウンロード

In [ ]:
# Clone TACO repo and download images
if not Path("TACO").exists():
    !git clone --depth 1 https://github.com/pedropro/TACO.git

os.chdir(WORKDIR / "TACO")
!python download.py
os.chdir(WORKDIR)

# Count downloaded images
count = len(list(Path("TACO/data").rglob("*.jpg"))) + len(list(Path("TACO/data").rglob("*.JPG"))) + len(list(Path("TACO/data").rglob("*.png")))
print(f"\nTACO images downloaded: {count}")

## 3. TACO → YOLO形式に変換（12クラス統合）

In [ ]:
# TACO category ID → consolidated class ID
CATEGORY_MAP = {
    4: 0, 5: 0, 7: 0,           # plastic_bottle
    10: 1, 11: 1, 12: 1, 50: 1,  # can
    20: 2, 22: 2,                 # paper_cup
    21: 3, 24: 3, 27: 3,          # plastic_cup
    6: 4, 26: 4, 23: 4,           # glass_bottle
    14: 5, 15: 5, 16: 5, 17: 5, 18: 5, 19: 5,  # cardboard
    25: 6,                         # food_waste
    30: 7, 31: 7, 33: 7, 34: 7, 56: 7,  # paper
    38: 8, 40: 8, 41: 8, 36: 8, 39: 8, 42: 8,  # plastic_bag
    45: 9, 46: 9, 47: 9, 43: 9, 44: 9, 49: 9, 55: 9, 29: 9,  # plastic_container
    1: 10,                         # battery
    57: 11,                        # styrofoam
}

CLASS_NAMES = [
    "plastic_bottle", "can", "paper_cup", "plastic_cup", "glass_bottle",
    "cardboard", "food_waste", "paper", "plastic_bag", "plastic_container",
    "battery", "styrofoam",
]

def convert_taco_to_yolo():
    taco_dir = Path("TACO/data")
    with open(taco_dir / "annotations.json") as f:
        coco = json.load(f)

    images = {img["id"]: img for img in coco["images"]}
    img_annotations = {}

    for ann in coco["annotations"]:
        if ann["category_id"] not in CATEGORY_MAP:
            continue
        img_id = ann["image_id"]
        if img_id not in img_annotations:
            img_annotations[img_id] = []

        yolo_cls = CATEGORY_MAP[ann["category_id"]]
        img_info = images[img_id]
        x, y, w, h = ann["bbox"]
        cx = (x + w / 2) / img_info["width"]
        cy = (y + h / 2) / img_info["height"]
        nw = w / img_info["width"]
        nh = h / img_info["height"]

        if nw * nh < 0.001:
            continue

        img_annotations[img_id].append((yolo_cls, cx, cy, nw, nh))

    available = []
    for img_id, anns in img_annotations.items():
        img_info = images[img_id]
        if (taco_dir / img_info["file_name"]).exists():
            available.append((img_id, img_info, anns))

    random.seed(42)
    random.shuffle(available)
    n = len(available)
    splits = {
        "train": available[:int(n * 0.8)],
        "val": available[int(n * 0.8):int(n * 0.9)],
        "test": available[int(n * 0.9):],
    }

    out_dir = Path("dataset")
    for split_name, split_data in splits.items():
        img_dir = out_dir / split_name / "images"
        lbl_dir = out_dir / split_name / "labels"
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)

        for img_id, img_info, anns in split_data:
            src = taco_dir / img_info["file_name"]
            shutil.copy2(src, img_dir / src.name)
            with open(lbl_dir / (src.stem + ".txt"), "w") as f:
                for cls, cx, cy, nw, nh in anns:
                    f.write(f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")

        print(f"  {split_name}: {len(split_data)} images")

    return out_dir

print("Converting TACO to YOLO format...")
dataset_dir = convert_taco_to_yolo()
print("Done!")

## 4. Open Images から不足クラスを補充

In [ ]:
# Download Open Images validation annotations
OI_BBOX_FILE = Path("oi_cache/validation-annotations-bbox.csv")
OI_BBOX_FILE.parent.mkdir(exist_ok=True)

if not OI_BBOX_FILE.exists():
    print("Downloading Open Images validation annotations...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",
        OI_BBOX_FILE
    )
    print("Done!")
else:
    print("Using cached annotations")

# Also try train annotations for more data
OI_TRAIN_FILE = Path("oi_cache/train-annotations-bbox.csv")
if not OI_TRAIN_FILE.exists():
    print("Downloading Open Images train annotations (~2.2GB, takes a few minutes)...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv",
        OI_TRAIN_FILE
    )
    print("Done!")
else:
    print("Using cached train annotations")

In [ ]:
# Open Images class mapping
OI_SUPPLEMENTS = {
    "/m/01c0z":  ("battery",      10, 400),
    "/m/02wbm":  ("food",          6, 400),
    "/m/02p5f1q": ("coffee_cup",    2, 300),
    "/m/089mxq": ("glass_bottle",   4, 250),
    "/m/02jnhm": ("tin_can",        1, 200),
    "/m/0641k":  ("paper",          7, 200),
    "/m/04kkgm": ("bowl_container", 9, 150),   # bowl → plastic_container
    "/m/07ptj3n": ("cup",           3, 200),   # cup → plastic_cup
}

def parse_oi_annotations(bbox_file, target_classes):
    """Parse Open Images annotations for target classes."""
    image_boxes = {}  # class_id -> {image_id -> [(x1,y1,x2,y2)]}
    for oi_id in target_classes:
        image_boxes[oi_id] = {}

    with open(bbox_file, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            label = row["LabelName"]
            if label in target_classes:
                img_id = row["ImageID"]
                if img_id not in image_boxes[label]:
                    image_boxes[label][img_id] = []
                image_boxes[label][img_id].append((
                    float(row["XMin"]), float(row["YMin"]),
                    float(row["XMax"]), float(row["YMax"]),
                ))
    return image_boxes

print("Parsing validation annotations...")
val_boxes = parse_oi_annotations(OI_BBOX_FILE, set(OI_SUPPLEMENTS.keys()))

print("Parsing train annotations (this takes ~1 min)...")
train_boxes = parse_oi_annotations(OI_TRAIN_FILE, set(OI_SUPPLEMENTS.keys()))

# Merge: train + validation
merged_boxes = {}
for oi_id in OI_SUPPLEMENTS:
    merged = {**train_boxes.get(oi_id, {}), **val_boxes.get(oi_id, {})}
    label = OI_SUPPLEMENTS[oi_id][0]
    merged_boxes[oi_id] = merged
    print(f"  {label}: {len(merged)} images available")

In [ ]:
def download_image(img_id, save_path):
    """Try to download an Open Images image."""
    urls = [
        f"https://storage.googleapis.com/openimages/validation/{img_id}.jpg",
        f"https://storage.googleapis.com/openimages/train/{img_id}.jpg",
    ]
    for url in urls:
        try:
            urllib.request.urlretrieve(url, save_path)
            return True
        except Exception:
            continue
    return False


def download_oi_class(oi_id, label, our_cls, limit, boxes_dict, out_dir):
    """Download images and create YOLO labels for one class."""
    img_dir = out_dir / "images"
    lbl_dir = out_dir / "labels"
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    image_ids = list(boxes_dict.keys())[:limit * 2]  # fetch extra in case of failures
    random.shuffle(image_ids)

    downloaded = 0
    for img_id in image_ids:
        if downloaded >= limit:
            break

        img_path = img_dir / f"oi_{label}_{img_id}.jpg"
        lbl_path = lbl_dir / f"oi_{label}_{img_id}.txt"

        if img_path.exists():
            downloaded += 1
            continue

        if download_image(img_id, img_path):
            with open(lbl_path, "w") as f:
                for x1, y1, x2, y2 in boxes_dict[img_id]:
                    cx = (x1 + x2) / 2
                    cy = (y1 + y2) / 2
                    w = x2 - x1
                    h = y2 - y1
                    f.write(f"{our_cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
            downloaded += 1

        if downloaded % 50 == 0 and downloaded > 0:
            print(f"    {label}: {downloaded}/{limit}")

    print(f"  {label}: {downloaded} images downloaded")
    return downloaded


# Download all supplement images
oi_out = Path("oi_supplements")
print("Downloading Open Images supplements...")
for oi_id, (label, cls_id, limit) in OI_SUPPLEMENTS.items():
    download_oi_class(oi_id, label, cls_id, limit, merged_boxes[oi_id], oi_out)

print("\nDone!")

In [ ]:
# Merge Open Images supplements into TACO train split
train_img = Path("dataset/train/images")
train_lbl = Path("dataset/train/labels")

oi_img = Path("oi_supplements/images")
oi_lbl = Path("oi_supplements/labels")

added = 0
if oi_img.exists():
    for img_file in oi_img.glob("*.*"):
        lbl_file = oi_lbl / (img_file.stem + ".txt")
        if lbl_file.exists():
            shutil.copy2(img_file, train_img / img_file.name)
            shutil.copy2(lbl_file, train_lbl / lbl_file.name)
            added += 1

print(f"Added {added} Open Images supplements to training set")

# Final class distribution
print("\nFinal class distribution (train split):")
counts = [0] * len(CLASS_NAMES)
for lbl_file in train_lbl.glob("*.txt"):
    with open(lbl_file) as f:
        for line in f:
            cls = int(line.split()[0])
            if cls < len(counts):
                counts[cls] += 1

for i, name in enumerate(CLASS_NAMES):
    bar = "█" * (counts[i] // 20)
    print(f"  {i:2d}: {name:20s} {counts[i]:5d}  {bar}")

print(f"\nTotal train images: {len(list(train_img.glob('*.*')))}")
print(f"Total train annotations: {sum(counts)}")

## 5. データセット設定ファイル作成

In [ ]:
# Write YOLO dataset config
dataset_dir = Path("dataset")
yaml_content = f"""path: {dataset_dir.resolve()}
train: train/images
val: val/images
test: test/images
nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""

with open(dataset_dir / "data.yaml", "w") as f:
    f.write(yaml_content)

print("Dataset config written to dataset/data.yaml")
print(yaml_content)

## 6. YOLO26n ファインチューニング (GPU)

T4 GPUで約15-20分で完了します。

In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLO26n
model = YOLO("yolo26n.pt")

# Fine-tune on waste dataset
results = model.train(
    data="dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,            # GPU
    patience=15,         # Early stopping
    warmup_epochs=5,
    mosaic=1.0,
    flipud=0.5,
    fliplr=0.5,
    mixup=0.15,
    scale=0.5,
    translate=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    name="waste-yolo26n-colab",
    exist_ok=True,
    verbose=True,
)

print(f"\nBest model: {results.save_dir}/weights/best.pt")

## 7. 結果の確認

In [ ]:
from IPython.display import Image, display

# Show training curves
results_img = Path("runs/detect/waste-yolo26n-colab/results.png")
if results_img.exists():
    display(Image(filename=str(results_img), width=800))

# Show confusion matrix
cm_img = Path("runs/detect/waste-yolo26n-colab/confusion_matrix.png")
if cm_img.exists():
    display(Image(filename=str(cm_img), width=600))

In [ ]:
# Validate on test set
best_model = YOLO("runs/detect/waste-yolo26n-colab/weights/best.pt")
metrics = best_model.val(data="dataset/data.yaml", split="test")

print(f"\n=== Test Set Results ===")
print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"\nPer-class AP50:")
for i, name in enumerate(CLASS_NAMES):
    if i < len(metrics.box.ap50):
        print(f"  {name:20s} {metrics.box.ap50[i]:.3f}")

## 8. ONNXエクスポート & ダウンロード

In [ ]:
# Export to ONNX
best_model.export(format="onnx", imgsz=640, simplify=True, opset=17)

onnx_path = Path("runs/detect/waste-yolo26n-colab/weights/best.onnx")
print(f"\nONNX model: {onnx_path}")
print(f"Size: {onnx_path.stat().st_size / 1024 / 1024:.1f} MB")
print(f"\nClass names: {best_model.names}")

In [ ]:
# Download the ONNX model
from google.colab import files

# Copy to a clean filename
shutil.copy2(onnx_path, "/content/yolo26n-waste.onnx")
files.download("/content/yolo26n-waste.onnx")

print("\n" + "=" * 50)
print("ダウンロード完了!")
print("このファイルを public/models/yolo26n.onnx に配置してください。")
print("=" * 50)

## 9. yolo-rules.json の更新版

ファインチューニング後のモデルは12クラスのゴミ専用クラスを使うため、
`public/models/yolo-rules.json` と `lib/yolo-inference.ts` の
クラス名配列も更新が必要です。

以下のセルを実行してコピー用のJSONを表示します。

In [ ]:
rules = {
    "confidence_threshold": 0.65,
    "min_box_area": 5000,
    "rules": {
        "plastic_bottle": {
            "itemName": "Plastic Bottle",
            "wasteStream": "recycling",
            "reasoning": "Plastic bottles are recyclable — empty and rinse before disposing.",
            "preAction": "Empty and rinse before recycling"
        },
        "can": {
            "itemName": "Can",
            "wasteStream": "recycling",
            "reasoning": "Metal cans are recyclable — rinse before disposing.",
            "preAction": "Rinse before recycling"
        },
        "paper_cup": {
            "itemName": "Paper Cup",
            "wasteStream": "landfill",
            "reasoning": "Most paper cups have a plastic lining and cannot be recycled.",
            "preAction": "Remove lid and sleeve before disposing"
        },
        "plastic_cup": {
            "itemName": "Plastic Cup",
            "wasteStream": "landfill",
            "reasoning": "Disposable plastic cups are not recyclable in most programs."
        },
        "glass_bottle": {
            "itemName": "Glass Bottle",
            "wasteStream": "recycling",
            "reasoning": "Glass bottles and jars are recyclable.",
            "preAction": "Rinse and remove caps"
        },
        "cardboard": {
            "itemName": "Cardboard",
            "wasteStream": "recycling",
            "reasoning": "Cardboard is recyclable — flatten before disposing.",
            "preAction": "Flatten before recycling"
        },
        "food_waste": {
            "itemName": "Food Waste",
            "wasteStream": "compost",
            "reasoning": "Food scraps are compostable."
        },
        "paper": {
            "itemName": "Paper",
            "wasteStream": "recycling",
            "reasoning": "Clean paper is recyclable. Soiled paper goes to compost."
        },
        "plastic_bag": {
            "itemName": "Plastic Bag",
            "wasteStream": "landfill",
            "reasoning": "Plastic bags jam sorting machinery and are not curbside recyclable."
        },
        "plastic_container": {
            "itemName": "Plastic Container",
            "wasteStream": "landfill",
            "reasoning": "Most disposable containers are mixed materials and not recyclable.",
            "preAction": "Empty contents before disposing"
        },
        "battery": {
            "itemName": "Battery",
            "wasteStream": "special",
            "reasoning": "Batteries contain hazardous materials and require special disposal."
        },
        "styrofoam": {
            "itemName": "Styrofoam",
            "wasteStream": "landfill",
            "reasoning": "Expanded polystyrene is not accepted in most recycling programs."
        }
    }
}

print(json.dumps(rules, indent=2))

# Also save to file for download
with open("/content/yolo-rules.json", "w") as f:
    json.dump(rules, f, indent=2)
files.download("/content/yolo-rules.json")